# BOLD post-validation recalibration comparison

## tl;dr

This notebook compares four bounded score-recalibration methods for both frozen BOLD scores using identical repeated patient-level folds. The winner is descriptive model updating, not independent external validation.

## Context & Methods

- Candidates: intercept-only, logistic intercept-plus-slope, isotonic, and a fixed 4-knot cubic spline.
- Reference: each unchanged score.
- Resampling: 20 repeats of five-fold patient-level cross-validation, stratified by source and outcome.
- Selection: lowest median pair-weighted log loss, then Brier score, then lower complexity.
- eICU-specific calibration is reported separately and cannot win the overall BOLD comparison.
- Coefficient refitting and threshold optimization are outside scope.

In [ ]:
import os
os.environ['MKL_THREADING_LAYER']='SEQUENTIAL'
os.environ['MKL_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
from pathlib import Path
import sys
PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'src').exists(): PROJECT_ROOT=PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from scripts.analysis.bold_recalibration_validation import run_validation
from scripts.qa.qa_bold_recalibration_validation import run_qa
ROOT=PROJECT_ROOT; RESULTS=ROOT/'bold_recalibration_validation'

## Data

Recreate all repeated-CV predictions and fixed-candidate comparisons from the frozen BOLD score file.

In [ ]:
result=run_validation(); pd.Series(result, name='value').to_frame()

## Results

### Overall BOLD selection table

In [ ]:
selection=pd.read_csv(RESULTS/'overall_selection_table.csv'); selection[['base_model','method','log_loss_median','brier_median','pr_auc_median','roc_auc_median','complexity_rank','selected']]

### Compare probability loss across all candidates

In [ ]:
summary=pd.read_csv(RESULTS/'recalibration_summary.csv'); plot=summary.query("scope=='overall_BOLD' and weighting=='pair'").copy(); plot['candidate']=plot['base_model']+' | '+plot['method']; plot=plot.sort_values('log_loss_median'); ax=plot.plot.barh(x='candidate',y='log_loss_median',legend=False,figsize=(8,5),color=['#2b6cb0' if m else '#a0aec0' for m in plot.method.ne('unchanged')]); ax.set_xlabel('Median cross-fitted log loss (lower is better)'); ax.set_ylabel(''); plt.tight_layout(); plt.show()

### eICU-specific secondary results and paired bootstrap uncertainty

In [ ]:
eicu=summary.query("scope=='eICU' and weighting=='pair'").sort_values('log_loss_median'); uncertainty=pd.read_csv(RESULTS/'selected_vs_unchanged_bootstrap.csv'); eicu[['base_model','method','log_loss_median','brier_median','pr_auc_median','roc_auc_median']], uncertainty

### Independent QA

In [ ]:
qa_result=run_qa(); qa=pd.read_csv(RESULTS/'independent_qa.csv'); qa_result, qa

## Takeaways

The selected candidate is the best *within this locked recalibration comparison*. Because BOLD outcomes were used to fit and select it, its cross-fitted performance estimates model-updating potential; it does not constitute a new external validation. A final recalibrator fit on all BOLD is saved for research use and must be evaluated on a third untouched cohort before any transport or deployment claim.